In [28]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [29]:
woeiv = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_iv.csv', index_col = [0]).reset_index().drop(columns = ['index'])

In [30]:
woeiv.head()

,features,iv,p_value,effect_size,finer_iv/exclusion_iv,finer_p_value/exclusion_p_value,finer_effect_size/exclusion_effect_size
0,person_income,0.429577,0.000000e+00,0.275793,0.802074,0.000000e+00,0.368958
1,loan_amnt,0.039353,1.180355e-49,0.082303,0.109290,7.534424e-126,0.142173
2,loan_int_rate,0.723749,0.000000e+00,0.372442,0.786965,0.000000e+00,0.386555
3,loan_percent_income,0.636250,0.000000e+00,0.349380,0.944489,0.000000e+00,0.438240
4,cb_person_cred_hist_length,0.003001,5.578833e-03,0.022559,0.003537,1.110083e-01,0.024474


In [31]:
spear = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/spearman.csv', index_col = [0])

In [32]:
spear.head()

,variable,spearman,pvalue
0,previous_loan_defaults_on_file,0.542635,0.0
1,loan_percent_income,0.321968,0.0
2,loan_int_rate,0.318893,0.0
3,person_income,0.265845,0.0
4,person_home_ownership,0.255919,0.0


In [33]:
LR = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_LR.csv', index_col=[0])

In [34]:
LR.head()

,features,pvalues
0,person_home_ownership,0.000000e+00
1,loan_percent_income,0.000000e+00
2,loan_int_rate,0.000000e+00
3,person_income,4.827454e-285
4,loan_intent,2.368039e-140


In [35]:
hoeff = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_hoeff.csv', index_col=[0])

In [36]:
hoeff.head()

,variable,hoeff
0,previous_loan_defaults_on_file,0.023814
1,loan_percent_income,0.013673
2,loan_int_rate,0.012947
3,person_income,0.008884
4,person_home_ownership,0.005867


In [37]:
vc = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_VC_Results.csv', index_col = [0])

In [38]:
vc.head()

,Cluster,Variable,RS_Own,RS_NC,RS_Ratio
0,0,person_emp_exp,0.911708,0.041047,0.092071
1,0,cb_person_cred_hist_length,0.911708,0.024536,0.090513
2,1,loan_amnt,0.798156,0.048339,0.212097
3,1,loan_percent_income,0.798156,0.047749,0.211965
4,2,previous_loan_defaults_on_file,1.000000,0.034715,0.000000


In [39]:
woeiv = woeiv.rename(columns = {'features':'variable', 'p_value':'iv_p_value'})

In [40]:
woeiv['iv_rank'] = woeiv['iv'].rank(ascending = False, method = 'dense')

In [41]:
woeiv = woeiv.sort_values(by = ['iv_rank']).reset_index().drop(columns = ['index'])

In [42]:
spear = spear.rename(columns = {'pvalue':'spear_pvalue'})

In [43]:
spear['spear_rank'] = spear['spearman'].rank(ascending = False, method = 'dense')

In [44]:
spear = spear.sort_values(by = ['spear_rank']).reset_index().drop(columns = ['index'])

In [45]:
LR = LR.rename(columns = {'features':'variable', 'pvalues':'LR_pvalues'})

In [46]:
LR['LR_rank'] = LR['LR_pvalues'].rank(ascending = True, method = 'dense')

In [47]:
LR = LR.sort_values(by = ['LR_rank']).reset_index().drop(columns = ['index'])

In [48]:
hoeff['hoeff_rank'] = hoeff['hoeff'].rank(ascending = False, method = 'dense')

In [49]:
hoeff = hoeff.sort_values(by = ['hoeff_rank']).reset_index().drop(columns = ['index'])

In [50]:
vc = vc.rename(columns = {'Variable':'variable'})

In [51]:
combine = woeiv.copy()

In [52]:
combine = combine.merge(spear, on = 'variable', how = 'left')

In [53]:
combine = combine.merge(LR, on = 'variable', how = 'left')

In [54]:
combine = combine.merge(hoeff, on = 'variable', how = 'left')

In [55]:
combine = combine.merge(vc, on = 'variable', how = 'left')

In [56]:
rf = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_Random Forest.csv', index_col=[0])
# rf2 = pd.read_csv('./Random Forest/Random Forest_FewCategorical.csv', index_col=[0])

In [57]:
rf = rf.rename(columns={'Feature':'variable'})
# rf2 = rf2.rename(columns={'Feature':'variable'})

In [58]:
xgb = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_XGBoost.csv', index_col=[0])
# xgb2 = pd.read_csv('./XGBoost/XGBoost_FewCategorical.csv', index_col=[0])

In [59]:
xgb = xgb.rename(columns={'Feature':'variable'})
# xgb2 = xgb2.rename(columns={'Feature':'variable'})

In [60]:
combine = combine.merge(rf, on='variable',how='left')
# combine = combine.merge(rf2, on='variable',how='left')

In [61]:
combine=combine.merge(xgb,on='variable',how='left')
# combine=combine.merge(xgb2,on='variable',how='left')

In [62]:
combine.head()

,variable,iv,iv_p_value,effect_size,finer_iv/exclusion_iv,finer_p_value/exclusion_p_value,finer_effect_size/exclusion_effect_size,iv_rank,spearman,spear_pvalue,spear_rank,LR_pvalues,LR_rank,hoeff,hoeff_rank,Cluster,RS_Own,RS_NC,RS_Ratio,RF_Importance,RF_Rank,XGB_Importance,XGB_Rank
0,previous_loan_defaults_on_file,inf,0.0,0.542561,inf,0.0,0.542561,1.0,0.542635,0.0,1.0,9.991543e-01,9.0,0.023814,1.0,2,1.000000,0.034715,0.000000,0.363909,1,0.856242,1
1,loan_int_rate,0.723749,0.0,0.372442,0.786965,0.0,0.386555,2.0,0.318893,0.0,3.0,0.000000e+00,1.0,0.012947,3.0,8,1.000000,0.034715,0.000000,0.169761,3,0.019474,4
2,loan_percent_income,0.636250,0.0,0.349380,0.944489,0.0,0.438240,3.0,0.321968,0.0,2.0,0.000000e+00,1.0,0.013673,2.0,1,0.798156,0.047749,0.211965,0.209079,2,0.038059,2
3,person_income,0.429577,0.0,0.275793,0.802074,0.0,0.368958,4.0,0.265845,0.0,4.0,4.827454e-285,2.0,0.008884,4.0,5,1.000000,0.029837,0.000000,0.095146,4,0.013230,6
4,person_home_ownership,0.428721,0.0,0.258080,0.428590,0.0,0.258073,5.0,0.255919,0.0,5.0,0.000000e+00,1.0,0.005867,5.0,6,1.000000,0.027057,0.000000,0.090585,5,0.033033,3


In [63]:
combine.to_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_variable_selection_combined_results.csv')

## Final Feature Selection Logic


To create a parsimonious and non-redundant feature set, I combined the outputs of all selection methods using the following strategy:

Top IV per Cluster: For each cluster identified in variable clustering, I selected the feature with the highest Information Value (IV) to avoid redundant variables with similar signals.

Top 30% Composite Features: Independently, I ranked all features using a normalized composite score that averaged across:

+ IV rank

+ Spearman correlation

+ Logistic regression p-values

+ Hoeffding's D statistic

+ Random Forest importance

+ XGBoost importance

Merge & Deduplicate: I merged both sets, then removed duplicates by keeping the feature with the stronger IV in case of conflict.

This approach balances predictive power (via multiple metrics) with variable diversity (by clustering), ensuring robust, interpretable inputs for downstream modeling.

Final selected features: 6 out of 11, automatically pruned for overlap and signal strength.

In [75]:
combined_df = combine.copy()

combined_df["xgb_rank_norm"] = -combined_df["XGB_Rank"]
combined_df["rf_rank_norm"] = -combined_df["RF_Rank"]
combined_df["hoeffding_d_norm"] = -combined_df["hoeff_rank"]
combined_df["spearman_corr_norm"] = -combined_df["spear_rank"]
combined_df["lr_pvalue_norm"] = -combined_df["LR_rank"]
combined_df["iv_norm"] = -combined_df["iv_rank"]

combined_df["composite_score"] = combined_df[
    ["xgb_rank_norm", "rf_rank_norm", "hoeffding_d_norm", "spearman_corr_norm", "lr_pvalue_norm", "iv_norm"]
].mean(axis=1)

top_iv_per_cluster = combined_df.sort_values("iv", ascending=False).drop_duplicates("Cluster")

top_30_cutoff = combined_df["composite_score"].quantile(0.7)
top_by_composite = combined_df[combined_df["composite_score"] >= top_30_cutoff]

final_features = pd.concat([top_iv_per_cluster, top_by_composite]).drop_duplicates("variable")

final_features = final_features[final_features["iv"] >= 0.02]

final_features.to_csv("final_selected_features.csv", index=False)

print(f"Selected {final_features.shape[0]} final variables")


Selected 6 final variables


In [76]:
final_features

,variable,iv,iv_p_value,effect_size,finer_iv/exclusion_iv,finer_p_value/exclusion_p_value,finer_effect_size/exclusion_effect_size,iv_rank,spearman,spear_pvalue,spear_rank,LR_pvalues,LR_rank,hoeff,hoeff_rank,Cluster,RS_Own,RS_NC,RS_Ratio,RF_Importance,RF_Rank,XGB_Importance,XGB_Rank,xgb_rank_norm,rf_rank_norm,hoeffding_d_norm,spearman_corr_norm,lr_pvalue_norm,iv_norm,composite_score
0,previous_loan_defaults_on_file,inf,0.000000e+00,0.542561,inf,0.000000e+00,0.542561,1.0,0.542635,0.000000e+00,1.0,9.991543e-01,9.0,0.023814,1.0,2,1.000000,0.034715,0.000000e+00,0.363909,1,0.856242,1,-1,-1,-1.0,-1.0,-9.0,-1.0,-2.333333
1,loan_int_rate,0.723749,0.000000e+00,0.372442,0.786965,0.000000e+00,0.386555,2.0,0.318893,0.000000e+00,3.0,0.000000e+00,1.0,0.012947,3.0,8,1.000000,0.034715,0.000000e+00,0.169761,3,0.019474,4,-4,-3,-3.0,-3.0,-1.0,-2.0,-2.666667
2,loan_percent_income,0.636250,0.000000e+00,0.349380,0.944489,0.000000e+00,0.438240,3.0,0.321968,0.000000e+00,2.0,0.000000e+00,1.0,0.013673,2.0,1,0.798156,0.047749,2.119652e-01,0.209079,2,0.038059,2,-2,-2,-2.0,-2.0,-1.0,-3.0,-2.000000
3,person_income,0.429577,0.000000e+00,0.275793,0.802074,0.000000e+00,0.368958,4.0,0.265845,0.000000e+00,4.0,4.827454e-285,2.0,0.008884,4.0,5,1.000000,0.029837,0.000000e+00,0.095146,4,0.013230,6,-6,-4,-4.0,-4.0,-2.0,-4.0,-4.000000
4,person_home_ownership,0.428721,0.000000e+00,0.258080,0.428590,0.000000e+00,0.258073,5.0,0.255919,0.000000e+00,5.0,0.000000e+00,1.0,0.005867,5.0,6,1.000000,0.027057,0.000000e+00,0.090585,5,0.033033,3,-3,-5,-5.0,-5.0,-1.0,-5.0,-4.000000
5,loan_intent,0.120147,1.192486e-138,0.141853,0.120147,1.192486e-138,0.141853,6.0,0.139045,1.433095e-139,6.0,2.368039e-140,3.0,0.002432,6.0,4,1.000000,0.006879,2.235827e-16,0.027761,6,0.013337,5,-5,-6,-6.0,-6.0,-3.0,-6.0,-5.333333
